In [1]:
import requests
from bs4 import BeautifulSoup
import re

headers = {"User-Agent": "Mozilla/5.0"}

def clean_title(title):
    title = re.sub(r"\(.*?\)", "", title)
    title = re.sub(r"\[.*?\]", "", title)
    title = re.sub(r"\bfeat\.?\b.*", "", title, flags=re.I)
    title = re.sub(r"featuring.*", "", title, flags=re.I)
    return title.strip()

def scrape_year(year):
    url = f"https://kworb.net/spotify/songs_{year}.html"

    response = requests.get(url, headers=headers)
    response.encoding = "utf-8"
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    songs = []

    table = soup.find("table")
    if table is None:
        return songs

    for row in table.find_all("tr")[1:]:
        cols = row.find_all("td")
    
        if len(cols) < 2:
            continue
    
        artist_title = cols[0].get_text(strip=True)
    
        # Second column contains the stream count
        streams = cols[1].get_text(strip=True).replace(",", "")
    
        if " - " in artist_title:
            artist, title = artist_title.split(" - ", 1)
    
            songs.append({
                "year": year,
                "artist": artist.strip(),
                "title": clean_title(title),
                "streams": int(streams)
            })

    return songs

In [2]:
all_songs = []

for year in range(2015, 2026):
    try:
        all_songs.extend(scrape_year(year))
    except requests.HTTPError:
        print(f"Could not retrieve {year}")

In [3]:
import pandas as pd

df = pd.DataFrame(all_songs)

pd.set_option('display.max_rows', None)
display(df)

df.to_csv("spotify_songs_2015_2025.csv", index=False, encoding="utf-8-sig") #if we need to export data as csv for other parts of the project

,year,artist,title,streams
0,2015,Lord Huron,The Night We Met,3858784170
1,2015,Justin Bieber,Love Yourself,3312873799
2,2015,Twenty One Pilots,Stressed Out,3118770904
3,2015,The Weeknd,The Hills,3112731294
4,2015,Justin Bieber,Sorry,3023709953
5,2015,Major Lazer,Lean On,2760278431
6,2015,Charlie Puth,We Don't Talk Anymore,2689454270
7,2015,Lukas Graham,7 Years,2578396178
8,2015,Shawn Mendes,Stitches,2573625181
9,2015,Tame Impala,The Less I Know The Better,2546245598


PermissionError: [Errno 13] Permission denied: 'spotify_songs_2015_2025.csv'